In [ ]:
from pathlib import Path
from metasmith.python_api import Agent, Source, SshSource, Std, DataInstanceLibrary, WorkflowTask
from local.constants import WORKSPACE_ROOT

dtypes, containers, transforms = Std()

agent_home = Source.FromLocal(Path("./cache/local_home").resolve())
# agent_home = SshSource("fir", Path("/scratch/phyberos/metasmith")).AsSource()
smith = Agent(
    home = agent_home,
)
# smith.Deploy()

In [ ]:
[k for k in dtypes.types if "reads" in k]

In [ ]:
inputs = DataInstanceLibrary("./cache/pointed_reads.xgdb")
inputs.AddItem(WORKSPACE_ROOT/"main/local_mock/cache/flye_in/lr_ss10.fastq", "std::hifi_reads")
inputs.AddItem(WORKSPACE_ROOT/"main/local_mock/cache/asm.xgdb/scadc.fna", "std::assembly")
inputs.Save()
for p, n, e in inputs.Iterate():
    print(n, p, e, e.parents)


# remote_root = Path("/project/6004975/phyberos/cyanoverse/main/logistics/interleave_test.xgdb")
# inputs = DataInstanceLibrary("./cache/remote_test.xgdb")
# inputs.AddItem(remote_root/"fwd.fq", "std::paired_reads_forward")
# inputs.AddItem(remote_root/"rev.fq", "std::paired_reads_reverse")
# inputs.Save()
# for p, n, e in inputs.Iterate():
#     print(n, p, e, e.parents)

In [ ]:
[k for k in dtypes.types if "cov" in k]

In [ ]:
_tasks = [
    smith.GenerateWorkflow(
        given      = [containers, inputs],
        transforms = [transforms],
        targets    = [dtypes[t]]
    )
    # for t in ["short_reads"]
    for t in ["per_contig_coverage"]
]

task = WorkflowTask.Merge((_tasks))
print(task.GetKey())
for step in [s for p in task.plans for s in p.steps]:
    print(step.order, step.transform.name)
# task.RenderDAG("./cache/dag")

In [ ]:
with open("./cache/slurm_account_fir") as f:
    SLURM_ACCOUNT = f.read()
task.config = dict(
    nextflow = dict(
        preset="slurm",
        slurm_account=SLURM_ACCOUNT,
        cpus=1,
        queueSize=200,
        memory='8 GB',
        time='3h',
    )
)

In [ ]:
smith.StageWorkflow(task, on_exist="clear")

In [ ]:
smith.RunWorkflow(task)

In [ ]:
bool("True".title()=="True")

In [ ]:
f"{True}"

In [ ]:
smith.CheckWorkflow(task)